In [4]:
from datasets import load_dataset

dataset = load_dataset("imdb")
print("Dataset loaded successfully")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset loaded successfully


In [5]:
import re
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_VOCAB_SIZE = 20000
BATCH_SIZE = 64
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"



In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text

def tokenize(text):
    return text.split()

train_texts = [tokenize(clean_text(x)) for x in dataset["train"]["text"]]
test_texts  = [tokenize(clean_text(x)) for x in dataset["test"]["text"]]

train_labels = dataset["train"]["label"]
test_labels  = dataset["test"]["label"]

counter = Counter()
for tokens in train_texts:
    counter.update(tokens)

vocab = [PAD_TOKEN, UNK_TOKEN] + [
    word for word, _ in counter.most_common(MAX_VOCAB_SIZE - 2)
]

word2idx = {word: idx for idx, word in enumerate(vocab)}

def numericalize(tokens):
    return [word2idx.get(token, word2idx[UNK_TOKEN]) for token in tokens]

train_sequences = [numericalize(tokens) for tokens in train_texts]
test_sequences  = [numericalize(tokens) for tokens in test_texts]


In [7]:
class IMDBDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx]), torch.tensor(self.labels[idx])

def collate_fn(batch):
    sequences, labels = zip(*batch)
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded.to(device), torch.tensor(labels).to(device)

train_loader = DataLoader(
    IMDBDataset(train_sequences, train_labels),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    IMDBDataset(test_sequences, test_labels),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


In [8]:
x, y = next(iter(train_loader))
print(x.shape, y.shape)


torch.Size([64, 741]) torch.Size([64])


In [9]:
# ===============================
# Common Setup
# ===============================
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = "/content/drive/MyDrive/Sentiment_Analysis_Project"
MODEL_DIR = f"{BASE_DIR}/models"

# Load dataset
dataset = load_dataset("imdb")


In [10]:
# ===============================
# Text Preprocessing
# ===============================
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.split()

train_tokens = [clean_text(x) for x in dataset["train"]["text"]]
test_tokens  = [clean_text(x) for x in dataset["test"]["text"]]

train_labels = dataset["train"]["label"]
test_labels  = dataset["test"]["label"]

# Vocabulary
MAX_VOCAB = 20000
counter = Counter()
for tokens in train_tokens:
    counter.update(tokens)

vocab = ["<pad>", "<unk>"] + [w for w, _ in counter.most_common(MAX_VOCAB-2)]
word2idx = {w:i for i, w in enumerate(vocab)}

def encode(tokens):
    return [word2idx.get(w, 1) for w in tokens]

train_seq = [encode(t) for t in train_tokens]
test_seq  = [encode(t) for t in test_tokens]

class IMDBDataset(Dataset):
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return torch.tensor(self.x[i]), torch.tensor(self.y[i])

def collate(batch):
    x, y = zip(*batch)
    x = pad_sequence(x, batch_first=True, padding_value=0)
    return x.to(device), torch.tensor(y).to(device)

train_loader = DataLoader(IMDBDataset(train_seq, train_labels), batch_size=64, shuffle=True, collate_fn=collate)
test_loader  = DataLoader(IMDBDataset(test_seq, test_labels),  batch_size=64, shuffle=False, collate_fn=collate)


In [11]:
# ===============================
# Custom LSTM Model
# ===============================
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])

model = SentimentLSTM(len(word2idx)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [13]:
# ===============================
# Train & Evaluate
# ===============================
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in tqdm(train_loader):
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            p = torch.argmax(model(x), 1)
            preds.extend(p.cpu())
            labels.extend(y.cpu())

    acc = accuracy_score(labels, preds)
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc:.4f}")

# Save model
import os
os.makedirs(MODEL_DIR, exist_ok=True)

torch.save(model.state_dict(), f"{MODEL_DIR}/custom_lstm.pth")
print("Custom LSTM saved")


100%|██████████| 391/391 [50:13<00:00,  7.71s/it]


Epoch 1 | Loss: 0.6873 | Acc: 0.5010


100%|██████████| 391/391 [44:29<00:00,  6.83s/it]


Epoch 2 | Loss: 0.6863 | Acc: 0.5010


100%|██████████| 391/391 [45:16<00:00,  6.95s/it]


Epoch 3 | Loss: 0.6859 | Acc: 0.5009


100%|██████████| 391/391 [46:53<00:00,  7.20s/it]


Epoch 4 | Loss: 0.6856 | Acc: 0.5012


100%|██████████| 391/391 [48:35<00:00,  7.46s/it]


Epoch 5 | Loss: 0.6853 | Acc: 0.5012
Custom LSTM saved
